# 3D Engine — TRELLIS.2 → GLB

**Purpose:** generate any 3D asset from a reference image.

This engine stops at 3D. A chair, building, prop, etc. can go directly to Unreal.
Only a character you later choose manually needs to enter the separate Animation Engine.

**Recommended Colab runtime:** A100 or another NVIDIA GPU with **24 GB+ VRAM**, matching the official TRELLIS.2 requirement.

This notebook deliberately uses **TRELLIS.2 directly** — no ComfyUI.

In [ ]:
# Check the assigned Colab GPU before installing anything.
!nvidia-smi

In [ ]:
# Get the small helper scripts from this repository.
import os, shutil, pathlib

if pathlib.Path("/content/My-works").exists():
    shutil.rmtree("/content/My-works")
!git clone -q --depth 1 https://github.com/Logan17de/My-works.git /content/My-works

TOOLS = "/content/My-works/ai-3d-animation-engines/3d-engine"
print("Helpers:", TOOLS)

## Install TRELLIS.2

A fresh Colab VM is disposable, so the notebook installs the official stack each time.
The model weights are also allowed to download into the runtime cache each session.

In [ ]:
%%bash
set -euo pipefail

apt-get update -qq
apt-get install -y -qq git git-lfs build-essential cmake ninja-build wget ffmpeg

if [ ! -x /opt/conda/bin/conda ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -p /opt/conda
fi

source /opt/conda/etc/profile.d/conda.sh

rm -rf /content/TRELLIS.2
git clone -q -b main https://github.com/microsoft/TRELLIS.2.git /content/TRELLIS.2 --recursive

cd /content/TRELLIS.2

# The official setup defaults to PyTorch 2.6 + CUDA 12.4.
if [ -d /usr/local/cuda-12.4 ]; then
  export CUDA_HOME=/usr/local/cuda-12.4
elif [ -d /usr/local/cuda ]; then
  export CUDA_HOME=/usr/local/cuda
fi

. ./setup.sh --new-env --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm

echo "TRELLIS.2 environment installed."

In [ ]:
# Upload ONE reference image.
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one image.")

input_name = next(iter(uploaded))
INPUT_IMAGE = f"/content/{input_name}"
print("Input:", INPUT_IMAGE)

In [ ]:
# Generate 3D + PBR GLB + turntable preview.
import os, subprocess, pathlib, shlex

OUTPUT_DIR = "/content/trellis_outputs"
ASSET_NAME = "asset"

cmd = [
    "/opt/conda/bin/conda", "run", "-n", "trellis2",
    "python", f"{TOOLS}/run_trellis2.py",
    "--input", INPUT_IMAGE,
    "--output-dir", OUTPUT_DIR,
    "--name", ASSET_NAME,
    "--envmap", "/content/TRELLIS.2/assets/hdri/forest.exr",
    "--decimation-target", "1000000",
    "--texture-size", "4096",
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd="/content/TRELLIS.2", check=True)

GLB_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}.glb"
PREVIEW_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}_preview.mp4"
print("GLB:", GLB_PATH)
print("Preview:", PREVIEW_PATH)

In [ ]:
# Preview inside Colab.
from IPython.display import Video, display
display(Video(PREVIEW_PATH, embed=True))

In [ ]:
# Download the final GLB (or copy it to Drive in the optional cell below).
from google.colab import files
files.download(GLB_PATH)

## Optional — save outputs to Google Drive

In [ ]:
# Optional persistence.
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/AI-3D-Engine"
# !cp -f "$GLB_PATH" "$PREVIEW_PATH" "/content/drive/MyDrive/AI-3D-Engine/"